# 🕵️ SQL Murder Mystery — Investigación

**Team Challenge SQL · Parte I** — Alejandro Dietta

> *A crime has taken place and the detective needs your help. [...] You vaguely remember that the crime was a **murder** that occurred sometime on **Jan.15, 2018** and that it took place in **SQL City**.*

Juego creado por Joon Park y Cathy He ([mystery.knightlab.com](http://mystery.knightlab.com)). En este notebook se reproduce la investigación atacando la base de datos local `data/sql-murder-mystery.db` con las mismas queries usadas en la web.

## 0. Conexión a la base de datos

Se abre el fichero `.db` y se copia a una base de datos **en memoria**. Así, el `INSERT` final en la tabla `solution` no modifica el fichero versionado en el repositorio y el notebook se puede ejecutar tantas veces como se quiera.

La función `q()` ejecuta una query y devuelve el resultado como `DataFrame` para verlo en formato tabla.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", None)  # ver descripciones y transcripciones completas

DB_PATH = Path("data/sql-murder-mystery.db")

disk = sqlite3.connect(DB_PATH)
conn = sqlite3.connect(":memory:")
disk.backup(conn)
disk.close()


def q(sql: str) -> pd.DataFrame:
    """Ejecuta una query SELECT y devuelve el resultado como DataFrame."""
    return pd.read_sql_query(sql, conn)


q("SELECT name FROM sqlite_master WHERE type = 'table'")

,name
0,crime_scene_report
1,drivers_license
2,person
3,facebook_event_checkin
4,interview
5,get_fit_now_member
6,get_fit_now_check_in
7,income
8,solution


## 1. Informe de la escena del crimen

Sabemos tres cosas: fue un **asesinato** (`type = 'murder'`), el **15/01/2018** y en **SQL City**. En `crime_scene_report` la fecha se guarda como entero con formato `AAAAMMDD`, por eso va sin comillas.

In [ ]:
q("""
SELECT *
FROM crime_scene_report
WHERE date = 20180115
    AND type = 'murder'
    AND city = 'SQL City';
""")

**Pista:** hay **2 testigos**:
1. El primero vive en **la última casa de "Northwestern Dr"**.
2. La segunda se llama **Annabel** y vive en **"Franklin Ave"**.

## 2. Identificar a los testigos

Los datos personales están en `person`. La dirección se divide en `address_number` y `address_street_name`.

**Testigo 1:** "la última casa" = el número más alto de la calle → ordenamos de mayor a menor y nos quedamos con la primera fila.

In [ ]:
q("""
SELECT *
FROM person
WHERE address_street_name = 'Northwestern Dr'
ORDER BY address_number DESC
LIMIT 2;
""")

,id,name,license_id,address_number,address_street_name,ssn
0,14887,Morty Schapiro,118009,4919,Northwestern Dr,111564949


**Testigo 2:** solo conocemos el nombre de pila, así que usamos `LIKE 'Annabel%'` (empieza por *Annabel*).

In [ ]:
q("""
SELECT *
FROM person
WHERE address_street_name = 'Franklin Ave'
    AND name LIKE 'Annabel%';
""")

**Testigos identificados:**

| id | Nombre | Dirección |
|---|---|---|
| 14887 | Morty Schapiro | 4919 Northwestern Dr |
| 16371 | Annabel Miller | 103 Franklin Ave |

## 3. Declaraciones de los testigos

La tabla `interview` guarda las transcripciones por `person_id`. Hacemos `JOIN` con `person` para ver el nombre de quien declara.

In [ ]:
q("""
SELECT witness.name, testimony.transcript
FROM interview AS testimony
JOIN person AS witness ON witness.id = testimony.person_id
WHERE testimony.person_id IN (14887, 16371);
""")

**Pistas:**
- **Morty:** un **hombre** con bolsa de *Get Fit Now Gym*, nº de socio que **empieza por "48Z"** (solo socios **gold**), huyó en un coche con matrícula que **contiene "H42W"**.
- **Annabel:** reconoció al asesino de su gimnasio el **9 de enero** (2018).

## 4. Cruzar las pistas: gimnasio + check-in + carnet de conducir

Cada pista está en una tabla distinta, así que las encadenamos con `JOIN`:

| Pista | Tabla |
|---|---|
| Socio "48Z…", gold | `get_fit_now_member` |
| Fue al gimnasio el 09/01/2018 | `get_fit_now_check_in` |
| Une socio con carnet | `person` (`person_id` → `license_id`) |
| Matrícula con "H42W", hombre | `drivers_license` |

`LIKE '48Z%'` = *empieza por*; `LIKE '%H42W%'` = *contiene*.

In [ ]:
q("""
SELECT member.id AS membership_id, member.name, member.membership_status,
    checkin.check_in_date, license.plate_number, license.gender,
    suspect.id AS person_id
FROM get_fit_now_member AS member
JOIN get_fit_now_check_in AS checkin ON checkin.membership_id = member.id
JOIN person AS suspect ON suspect.id = member.person_id
JOIN drivers_license AS license ON license.id = suspect.license_id
WHERE member.id LIKE '48Z%'
    AND member.membership_status = 'gold'
    AND checkin.check_in_date = 20180109
    AND license.plate_number LIKE '%H42W%'
    AND license.gender = 'male';
""")

**Sospechoso:** solo una persona cumple todas las pistas → **Jeremy Bowers** (`person_id` 67318).

## 5. Comprobar la solución

El juego valida la respuesta insertando el nombre en la tabla `solution`: un *trigger* de la base de datos sustituye la fila por un mensaje indicando si es correcto. Como trabajamos sobre la copia en memoria, el fichero `.db` no se modifica.

In [ ]:
conn.execute("INSERT INTO solution VALUES (1, 'Jeremy Bowers')")
conn.commit()
q("SELECT value FROM solution")